# 単語ベクトル(word embedding)を体感する

このノートブックは、学習済みの単語ベクトル **GloVe**（単語埋め込み＝word embedding の代表例）を使って、
「ことばを数のベクトルで表す」と何ができるかを **手元で再現** します。

- 上から順にセルを実行するだけ。**Colab でもそのまま動きます**。
- 各図は **その場のコードで生成** します（保存済み画像の貼り付けではありません）。
- 図の単語ラベルは英語、説明は日本語です。

**実行方法**：メニューから「ランタイム → すべて実行」(Colab) または
「Run → Run All Cells」(Jupyter)。初回はベクトル(約376MB)をダウンロードするため数分かかります。
一度ダウンロードすれば **再ダウンロードは不要** ですが、カーネルを再起動するとメモリ上のベクトルは消えるため、
**ディスクからメモリへの読み込み（数十秒）は実行のたびに発生します**。

## 使用する単語ベクトルと出典

このノートは、Stanford の学習済み単語ベクトル **GloVe** を、`gensim` のダウンローダ経由で取得して使います。

- **モデル**：`glove-wiki-gigaword-300`（gensim-data から取得）
- **学習コーパス**（corpus＝学習に使う大量の文章）：Wikipedia 2014 ＋ Gigaword 5（約60億トークン〔token＝単語などの最小のかたまり〕、uncased＝すべて小文字）
- **規模**：400,000 語 × 300 次元（ファイル約 376 MB）
- **学習済みベクトルの配布元**：Stanford GloVe プロジェクト <https://nlp.stanford.edu/projects/glove/>
  ／ 取得に使った gensim-data <https://github.com/piskvorky/gensim-data>
- **論文（査読版・公式）**：Pennington, Socher, Manning.
  *GloVe: Global Vectors for Word Representation.* EMNLP 2014.
  ACL Anthology: <https://aclanthology.org/D14-1162/>（DOI: 10.3115/v1/D14-1162）

このノートに出てくる **図・数値はすべて、下のコードをその場で実行して得た手元の結果** です。単語ベクトルが示す傾向（後半の「バイアス」を含む）は、上記コーパス＝学習に使った文章の性質を反映したものです。関連文献（word2vec／バイアスの分析）はノート末尾の「出典」を参照してください。

<sub>※ ライセンス：GloVe の学習済みベクトルは Open Data Commons PDDL（<http://opendatacommons.org/licenses/pddl/>）で配布されています。</sub>

## 準備：ライブラリの読み込みとベクトルの取得先

最初に、必要なライブラリ（`gensim` / `scikit-learn` / `matplotlib` / `numpy`）を読み込みます。
Colab など未導入の環境では自動でインストールします。**このセルを最初に必ず実行してください**
（以降のセルは、ここで読み込んだ `np` / `plt` / `api` などをそのまま使います）。

単語ベクトル本体（約376MB）は、ホームの隠しキャッシュ `~/.cache/gensim-data/` に **一度だけダウンロード** され、以後はそこから読み込みます（HuggingFace などと同じ流儀。ノートをどこに置いても共通で、保存先は実行時に表示されます）。
再ダウンロードは起きませんが、**カーネルを再起動するとメモリ上のベクトルは消える** ので、次の `api.load(...)` でのディスク→メモリ読み込み（数十秒）は実行のたびに発生します。

実行すると各図は、画面表示に加えて `OUTPUT_DIR`（既定 `../outputs/`）に **PNG でも保存** されます。
保存先を変えたいときは下のセルの `OUTPUT_DIR` の1行を書き換えるだけ。
Colab など `../outputs/` に書けない環境では、自動でカレント直下の `outputs/` に切り替わります。

In [ ]:
# === セットアップ：ライブラリの読み込みとベクトルの置き場所 ===
# Colab など、ライブラリが無い環境では自動でインストールします。
import importlib, subprocess, sys

def _ensure(pip_name, import_name):
    try:
        importlib.import_module(import_name)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_name], check=True)

for _pip, _imp in [("gensim", "gensim"), ("scikit-learn", "sklearn"),
                   ("matplotlib", "matplotlib"), ("umap-learn", "umap")]:
    _ensure(_pip, _imp)

import os
# ベクトル(約376MB)のキャッシュ先。HuggingFace などと同じく、ホームの隠しキャッシュ
# ~/.cache/gensim-data に置く（ホームに見えるフォルダを作らず、ノートをどこに置いても共通）。
# ※ この設定は gensim.downloader を import する「前」に行う必要がある。
CACHE_DIR = os.path.join(os.environ.get("XDG_CACHE_HOME") or os.path.expanduser("~/.cache"),
                         "gensim-data")
os.makedirs(CACHE_DIR, exist_ok=True)
os.environ["GENSIM_DATA_DIR"] = CACHE_DIR

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from sklearn.decomposition import PCA
import gensim.downloader as api
%matplotlib inline

plt.rcParams["axes.unicode_minus"] = False
plt.rcParams["figure.dpi"] = 110

# 図の PNG 書き出し先。保存先を変えたいときは、この1行だけ変更すればよい。
# Colab は /content 直下の outputs/ に保存（見つけやすい）。ローカルは従来どおり ../outputs。
OUTPUT_DIR = "outputs" if "google.colab" in sys.modules else "../outputs"
# 保存ファイル名の接頭辞（スライド素材として識別しやすくする。変えたいときはここ1行）。
FILE_PREFIX = "wordvec_demo_"
os.makedirs(OUTPUT_DIR, exist_ok=True)

def finish(fig, name=None):
    """図を画面に表示し、name があれば OUTPUT_DIR に PNG としても保存する。"""
    if name:
        fig.savefig(os.path.join(OUTPUT_DIR, f"{FILE_PREFIX}{name}.png"), dpi=150, bbox_inches="tight")
    plt.show()

_cache_disp = CACHE_DIR.replace(os.path.expanduser("~"), "~")
print(f"セットアップ完了 (Python {sys.version.split()[0]})")
print(f"  ベクトルのキャッシュ先 : {_cache_disp}")
print(f"  図の保存先            : {OUTPUT_DIR}")

## まず：単語をベクトルにする、とは？

コンピュータはことばをそのままでは扱えないので、各単語を **数のベクトル**（数の並び）で表します。
このノートで使う GloVe では、**1単語 = 300個の数** です。

- **意味が近い ＝ ベクトルが近い**。近さは2つのベクトルの「向き」のそろい具合
  （**コサイン類似度**：1に近いほど似ていて、0なら無関係）で測ります。
- このベクトルは「ことばの意味の地図」のようなもので、**足し算・引き算** にも意味が出ます。
  有名な例が `king − man + woman ≈ queen`（王 − 男 + 女 ≈ 女王）。

まずは、1単語がただの数の並びであることと、近い単語を実際に見てみましょう。

In [ ]:
# GloVe ベクトルを読み込む（初回はダウンロード、以降はキャッシュ）
MODEL = "glove-wiki-gigaword-300"
kv = api.load(MODEL)

# 実際に読み込んだベクトルファイル（場所・サイズ）を表示
_vec_file = os.path.join(CACHE_DIR, MODEL, MODEL + ".gz")
print(f"モデル: {MODEL}")
if os.path.exists(_vec_file):
    _vec_disp = _vec_file.replace(os.path.expanduser("~"), "~")
    print(f"  ファイル: {_vec_disp}  ({os.path.getsize(_vec_file) / 1024 / 1024:.0f} MB)")
print(f"  語彙数 = {len(kv.index_to_key):,} 語,  1単語の次元 = {kv.vector_size}")

# 1単語は「数の並び」にすぎない（身近な例として cat）
v = kv["cat"]
print(f"\n'cat' のベクトル: 形 = {v.shape},  最初の5次元 = {v[:5].round(3)}")

# コサイン類似度が高い（=近い）単語を、いろいろな単語で見てみる
print("\nいくつかの単語に「近い」単語（コサイン類似度 上位5）:")
for w in ["cat", "dog", "tokyo", "japan", "king", "red",
          "music", "computer", "coffee", "soccer"]:
    neigh = ", ".join(x for x, _ in kv.most_similar(w, topn=5))
    print(f"  {w:9s} → {neigh}")

## 1. 単語はベクトル、近いベクトルは関連語

**結論**：単語をベクトルにすると、意味的に関連する語が「近く」に来る（コサイン類似度が高い）。
`cat` には dog / feline（ネコ科）など動物語が、`tokyo` には japan / seoul など地名が近い。

In [ ]:
examples = ["cat", "tokyo"]
fig, axes = plt.subplots(1, len(examples), figsize=(11, 4.2))
for ax, q in zip(axes, examples):
    pairs = kv.most_similar(q, topn=8)
    words = [w for w, _ in pairs][::-1]   # 下から上へ並べる
    sims = [s for _, s in pairs][::-1]
    ax.barh(words, sims, color="#4C72B0")
    ax.set_xlim(0, 1)
    ax.set_xlabel("cosine similarity")
    ax.set_title(f"nearest words to  “{q}”")
    for y, s in enumerate(sims):
        ax.text(s + 0.01, y, f"{s:.2f}", va="center", fontsize=8, color="0.3")
fig.suptitle("A word is a vector; nearby vectors = related meanings", fontsize=12)
fig.tight_layout(rect=(0, 0, 1, 0.96))
finish(fig, "01_nearest")

### コサイン類似度 ＝ 2本のベクトルのなす角

ここで使う「近さ」は **コサイン類似度**＝2本のベクトルの **なす角 θ のコサイン** です（内積の定義から）：

$$\cos\theta=\frac{\mathbf{x}\cdot\mathbf{y}}{\|\mathbf{x}\|\,\|\mathbf{y}\|}$$

- 向きが同じ（θ=0°）→ cos = 1（最も似ている）／ 直角（θ=90°）→ cos = 0（無関係）／ 逆向き → cos = −1。
- ベクトルは300次元ですが、**2本だけなら、そのなす角を平面に“誤差なし”で描けます**。
- 関連語ほど角が小さく cos が大きい。無関係な語はほぼ直角で cos ≈ 0——
  だから movie–film の **0.86 は「とても似ている」** と分かります（無関係なら 0 付近）。

In [ ]:
# コサイン類似度 = 2本のベクトルのなす角 θ（長さは元のノルムのまま＝正規化しない）
def plot_cos(ax, w1, w2):
    a, b = kv[w1], kv[w2]
    cos = float(kv.similarity(w1, w2))
    theta = np.arccos(np.clip(cos, -1.0, 1.0))
    na, nb = np.linalg.norm(a), np.linalg.norm(b)          # 元のベクトルの長さ（ノルム）
    ang1, ang2 = np.pi/2 + theta/2, np.pi/2 - theta/2      # 上向きの V 字（なす角は正確）
    p1 = na * np.array([np.cos(ang1), np.sin(ang1)])        # 長さ = ‖cat‖（正規化しない）
    p2 = nb * np.array([np.cos(ang2), np.sin(ang2)])
    for p, w, nrm in [(p1, w1, na), (p2, w2, nb)]:
        ax.annotate("", xy=p, xytext=(0, 0),
                    arrowprops=dict(arrowstyle="-|>", color="#4C72B0", lw=2.6))
        ax.annotate(w, p, fontsize=14, fontweight="bold", color="#27406b",
                    xytext=(7 if p[0] >= 0 else -7, 7), textcoords="offset points",
                    ha="left" if p[0] >= 0 else "right")
        mid = p * 0.5
        ax.text(mid[0] + (1.0 if p[0] >= 0 else -1.0), mid[1],
                rf"$\|\mathrm{{{w}}}\|={nrm:.1f}$", fontsize=10, color="0.45",
                ha="left" if p[0] >= 0 else "right", va="center")
    r = 0.28 * min(na, nb)
    ax.add_patch(mpatches.Arc((0, 0), 2 * r, 2 * r, angle=0,
                              theta1=np.degrees(ang2), theta2=np.degrees(ang1),
                              color="#C44E52", lw=2.5))
    ax.text(0, r * 1.45, "θ", color="#C44E52", fontsize=15, ha="center", va="bottom")
    ax.set_title(f"{w1} – {w2}", fontsize=14)
    ax.text(0.5, -0.04, f"cos θ = {cos:.2f}   (θ = {np.degrees(theta):.0f}°)",
            transform=ax.transAxes, ha="center", fontsize=12, color="0.2")
    m = max(na, nb) * 1.30
    ax.set_xlim(-m, m); ax.set_ylim(-0.12 * m, m * 1.08)
    ax.set_aspect("equal"); ax.axis("off")

fig, axes = plt.subplots(1, 2, figsize=(11, 5.2))
plot_cos(axes[0], "movie", "film")    # 同義語：角がとても小さい → cos が大きい
plot_cos(axes[1], "movie", "banana")  # 無関係語：ほぼ直角 → cos がほぼ 0
fig.suptitle(r"cosine similarity  $\cos\theta=\dfrac{\mathbf{x}\cdot\mathbf{y}}{\|\mathbf{x}\|\,\|\mathbf{y}\|}$", fontsize=15)
fig.tight_layout(rect=(0, 0, 1, 0.90))
finish(fig, "02_cosine_angle")

他の単語でも見てみましょう。下は **4セット**で、各セットとも **左＝似た語ペア (a,b)／右＝無関係な語ペア (a,c)**。似ているほど角が小さく、無関係だと直角に近いことが分かります。

In [ ]:
# （2ページ目）1セット = 三つ組(a,b,c) の [(a,b) 似てる | (a,c) 無関係]。4セットを 2x2（計8図）
SETS = [
    ("cat",   "dog",   "car"),     # 動物
    ("king",  "queen", "banana"),  # 王位
    ("tokyo", "osaka", "banana"),  # 都市
    ("car",   "truck", "banana"),  # 乗り物
]
def pair_pts(w1, w2):
    cos = float(kv.similarity(w1, w2))
    theta = np.arccos(np.clip(cos, -1.0, 1.0))
    na, nb = np.linalg.norm(kv[w1]), np.linalg.norm(kv[w2])
    a1, a2 = np.pi/2 + theta/2, np.pi/2 - theta/2          # 上向き V（なす角は正確・長さは元のノルム）
    return dict(cos=cos, na=na, nb=nb, a1=a1, a2=a2,
                p1=na*np.array([np.cos(a1), np.sin(a1)]),
                p2=nb*np.array([np.cos(a2), np.sin(a2)]))

def draw(ax, w1, w2, d, xlim, ylim, cos_y):
    for p, w in [(d["p1"], w1), (d["p2"], w2)]:
        ax.annotate("", xy=p, xytext=(0, 0),
                    arrowprops=dict(arrowstyle="-|>", color="#4C72B0", lw=3.3, mutation_scale=18))
        ax.annotate(w, p, fontsize=11, fontweight="bold", color="#27406b",
                    xytext=(4 if p[0] >= 0 else -4, 4), textcoords="offset points",
                    ha="left" if p[0] >= 0 else "right")
    r = 0.32 * min(d["na"], d["nb"])
    ax.add_patch(mpatches.Arc((0, 0), 2*r, 2*r, angle=0,
                              theta1=np.degrees(d["a2"]), theta2=np.degrees(d["a1"]),
                              color="#C44E52", lw=3.0))
    ax.text(0, cos_y, f"cos = {d['cos']:.2f}", ha="center", va="top",
            fontsize=11, color="0.15")          # 原点のすぐ下（左右で同じ高さ）
    ax.set_xlim(*xlim); ax.set_ylim(*ylim); ax.set_aspect("equal"); ax.axis("off")

fig = plt.figure(figsize=(18, 7))
subfigs = fig.subfigures(2, 2, wspace=0.03, hspace=0.18)
for sf, (a, b, c) in zip(subfigs.flat, SETS):
    d1, d2 = pair_pts(a, b), pair_pts(a, c)
    pts = [d1["p1"], d1["p2"], d2["p1"], d2["p2"]]
    xr = max(abs(p[0]) for p in pts); yt = max(p[1] for p in pts)
    M = max(d1["na"], d1["nb"], d2["na"], d2["nb"])
    xlim = (-(xr + 0.30*M), xr + 0.30*M)        # 左右パネルで共有＝同縮尺・原点の高さも一致
    ylim = (-0.22*yt, yt*1.16); cos_y = -0.09*yt
    axs = sf.subplots(1, 2)
    draw(axs[0], a, b, d1, xlim, ylim, cos_y)   # 似てる
    draw(axs[1], a, c, d2, xlim, ylim, cos_y)   # 無関係
    sf.suptitle(f"{a} :  {b} (similar)   vs   {c} (unrelated)",
                fontsize=12, fontweight="bold")
finish(fig, "03_cosine_gallery")

## 2. 意味は幾何構造をもつ（カテゴリごとに固まる）

**結論**：国・動物・食べ物・動詞という4カテゴリの単語を2次元に圧縮（PCA＝主成分分析。情報を保ちつつ次元を2に減らす手法）すると、
**同じカテゴリの語が一カ所に集まる**。意味が空間の「位置」になっている。
（見どころ：動詞の `cook` は料理＝食べ物カテゴリの近くに引っ張られる。）

In [ ]:
CATEGORIES = {
    "countries": ["japan", "china", "korea", "india", "france", "germany",
                  "italy", "brazil", "russia"],
    "animals":   ["cat", "dog", "horse", "elephant", "tiger", "rabbit",
                  "monkey", "lion", "wolf"],
    "foods":     ["pizza", "sushi", "bread", "rice", "cheese", "banana",
                  "coffee", "noodle", "curry"],
    "verbs":     ["run", "walk", "jump", "sing", "write", "read", "swim",
                  "dance", "cook"],
}
COLORS = {"countries": "#4C72B0", "animals": "#55A868",
          "foods": "#C44E52", "verbs": "#8172B3"}

words, labels = [], []
for cat, ws in CATEGORIES.items():
    for w in ws:
        words.append(w); labels.append(cat)
X = np.array([kv[w] for w in words])
P = PCA(n_components=2, random_state=0).fit_transform(X)

fig, ax = plt.subplots(figsize=(8.5, 6.5))
for cat in CATEGORIES:
    idx = [i for i, l in enumerate(labels) if l == cat]
    ax.scatter(P[idx, 0], P[idx, 1], s=70, color=COLORS[cat], label=cat, zorder=3)
for i, w in enumerate(words):
    ax.annotate(w, P[i], fontsize=9, xytext=(4, 4),
                textcoords="offset points", color="0.25")
ax.legend(loc="best", title="category")
ax.set_title("Words group by meaning — PCA (2D)", fontsize=13)
ax.set_xlabel("PCA dim 1"); ax.set_ylabel("PCA dim 2")
ax.grid(alpha=0.25)
fig.tight_layout()
finish(fig, "04_clusters")

## 補足①：次元削減を“雰囲気”でつかむ

§2 では 300次元のベクトルを **PCA で2次元** に落として「カテゴリが固まる」のを見ました。
ここでは次元削減そのものを、もう少しだけ見てみます（数式は気にせず雰囲気でOK）。
**PCA**（線形）と **t-SNE / UMAP**（非線形）を順に試します。

### PCA（principal component analysis ＝ 主成分分析）

**PCA** は、データのばらつきが大きい方向から順に新しい軸（主成分）をとり、少ない次元で表す **線形** の手法です。

**PCA の「中身」**：300本の軸を「情報量（ばらつき）の大きい順」に並べ替えます。
各軸がどれだけ情報を説明するか（**寄与率**＝その軸の固有値の割合）を見ると、2次元で何割残るかが分かります。
（実装・詳細：scikit-learn ドキュメント <https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.PCA.html>）

In [ ]:
# PCA の「中身」：各主成分の寄与率（固有値の割合）と累積
words = [w for ws in CATEGORIES.values() for w in ws]
X = np.array([kv[w] for w in words])
_pca = PCA(n_components=10, random_state=0).fit(X)
ratio = _pca.explained_variance_ratio_ * 100      # 寄与率(%) = 固有値の割合
print("主成分    寄与率     累積寄与率")
cum = 0.0
for i, r in enumerate(ratio, 1):
    cum += r
    print(f"  PC{i:<2d}   {r:5.1f}%     {cum:5.1f}%")
print(f"\n→ 2次元（PC1+PC2）では約 {ratio[:2].sum():.0f}% しか説明できない＝2D散布図は『だいたいの雰囲気』。")

In [ ]:
# 同じ4カテゴリを 3次元 PCA（PC1–PC2–PC3）で
words, labels = [], []
for cat, ws in CATEGORIES.items():
    for w in ws:
        words.append(w); labels.append(cat)
P3 = PCA(n_components=3, random_state=0).fit_transform(np.array([kv[w] for w in words]))

fig = plt.figure(figsize=(8, 6.6))
ax = fig.add_subplot(projection="3d")
for cat in CATEGORIES:
    idx = [i for i, l in enumerate(labels) if l == cat]
    ax.scatter(P3[idx, 0], P3[idx, 1], P3[idx, 2], s=45,
               color=COLORS[cat], label=cat, depthshade=False)
for i, w in enumerate(words):                       # 2D版と同じく各点に単語ラベル
    ax.text(P3[i, 0], P3[i, 1], P3[i, 2], w, fontsize=7, color="0.25")
ax.set_xlabel("PC1"); ax.set_ylabel("PC2"); ax.set_zlabel("PC3")
ax.set_title("Words group by meaning — PCA (3D)", fontsize=13)
ax.legend(loc="upper left", fontsize=9)
ax.view_init(elev=18, azim=-60)
ax.set_box_aspect(None, zoom=0.85)   # 余白を詰めつつ PC3 ラベルを収める
finish(fig, "05_clusters_3d")

**コサインは長さに依存しません**（角度だけ）が、**PCA / t-SNE は距離＝長さに依存**します。コサインに合わせたいなら、先に **長さ1へ正規化**してから次元削減する手もあります。下は raw と 正規化後の PCA の比較——ふつうの語は頻度が近く長さも近いので **ほとんど変わりません**。

In [ ]:
# 参考：長さ1に正規化してから PCA（コサイン基準）。raw と比べてほぼ同じ
words, labels = [], []
for cat, ws in CATEGORIES.items():
    for w in ws:
        words.append(w); labels.append(cat)
X = np.array([kv[w] for w in words])
Xn = X / np.linalg.norm(X, axis=1, keepdims=True)        # 各ベクトルを長さ1へ

P_raw  = PCA(n_components=2, random_state=0).fit_transform(X)
P_norm = PCA(n_components=2, random_state=0).fit_transform(Xn)
for k in range(2):                                        # PCA の符号は任意なので向きを揃える
    if np.corrcoef(P_raw[:, k], P_norm[:, k])[0, 1] < 0:
        P_norm[:, k] = -P_norm[:, k]

fig, axes = plt.subplots(1, 2, figsize=(13, 6))
for ax, title, P in [(axes[0], "raw vectors", P_raw),
                     (axes[1], "L2-normalized (length 1)", P_norm)]:
    for cat in CATEGORIES:
        idx = [i for i, l in enumerate(labels) if l == cat]
        ax.scatter(P[idx, 0], P[idx, 1], s=55, color=COLORS[cat], label=cat, zorder=3)
    for i, w in enumerate(words):
        ax.annotate(w, P[i], fontsize=7, xytext=(3, 3),
                    textcoords="offset points", color="0.3")
    ax.set_title(f"PCA — {title}", fontsize=12)
    ax.set_xlabel("PC1"); ax.set_ylabel("PC2"); ax.grid(alpha=0.25)
axes[0].legend(loc="best", fontsize=8)
fig.suptitle("Cosine ignores length; PCA uses distance. Normalizing to length 1 first "
             "makes PCA cosine-consistent — little change for common words", fontsize=12)
fig.tight_layout(rect=(0, 0, 1, 0.94))
finish(fig, "06_pca_raw_vs_normalized")

### 非線形の次元削減：t-SNE と UMAP

PCA は「まっすぐな（線形の）押しつぶし」でした。**t-SNE / UMAP** は曲げながら畳む非線形の方法で、
**近いものを近くに集める**のが得意です（同じカテゴリがよりくっきり分かれて見えやすい）。

- **t-SNE**（t-distributed Stochastic Neighbor Embedding）：近い点どうしの“近さ”を保つように低次元へ写す手法。
  van der Maaten & Hinton, JMLR 2008 — <https://www.jmlr.org/papers/v9/vandermaaten08a.html>。
  Distill「How to Use t-SNE Effectively」<https://distill.pub/2016/misread-tsne/> は、
  **スライダーを動かして見え方の変化を試せる**解説で、t-SNE の癖が直感的につかめます。
- **UMAP**（Uniform Manifold Approximation and Projection）：t-SNE に似た非線形手法で、より高速・大規模データ向き。
  McInnes, Healy, Melville, arXiv:1802.03426 — <https://arxiv.org/abs/1802.03426>、実装ドキュメント <https://umap-learn.readthedocs.io/>。

ただし注意：これらの図は **点どうしの距離・かたまりの大きさ・形は“そのままの意味”をもちません**。
乱数や設定（perplexity / n_neighbors）でも見た目が変わります。
読むのは「**同じ種類が固まるか**」くらいの“雰囲気”だけにしましょう。

In [ ]:
# 非線形の2次元化：t-SNE と UMAP（それぞれ別の図にする）
import warnings
from sklearn.manifold import TSNE

words, labels = [], []
for cat, ws in CATEGORIES.items():
    for w in ws:
        words.append(w); labels.append(cat)
X = np.array([kv[w] for w in words])

with warnings.catch_warnings():            # umap/t-SNE の細かな警告は隠す（学生向け・パス混入も防ぐ）
    warnings.simplefilter("ignore")
    import umap
    embeds = {
        "t-SNE": TSNE(n_components=2, perplexity=8, init="pca",
                      random_state=0).fit_transform(X),
        "UMAP":  umap.UMAP(n_components=2, n_neighbors=10, min_dist=0.3,
                           random_state=0).fit_transform(X),
    }

def plot_embed(name, P, save):
    fig, ax = plt.subplots(figsize=(7.6, 6.2))
    for cat in CATEGORIES:
        idx = [i for i, l in enumerate(labels) if l == cat]
        ax.scatter(P[idx, 0], P[idx, 1], s=60, color=COLORS[cat], label=cat)
    for i, w in enumerate(words):
        ax.annotate(w, P[i], fontsize=8, xytext=(3, 3),
                    textcoords="offset points", color="0.3")
    ax.set_title(f"Words group by meaning — {name} (2D)", fontsize=13)
    ax.set_xticks([]); ax.set_yticks([])
    ax.legend(loc="best", fontsize=9)
    fig.tight_layout()
    finish(fig, save)

plot_embed("t-SNE", embeds["t-SNE"], "07_tsne")
plot_embed("UMAP", embeds["UMAP"], "08_umap")

#### （さらに脱線）点を増やすと t-SNE / UMAP の本領が見える

上の4カテゴリ（36語）くらいだと、PCA でも t-SNE/UMAP でも似たような図になりがちです。
語数を **約5倍（~160語・10カテゴリ）** に増やして“ごちゃごちゃ”させると、違いが出ます：
**PCA は重なって団子になりやすい**のに対し、**t-SNE / UMAP は同じ種類をくっきり島（クラスタ）に分けます**——これが非線形手法の本領です。
（ただし注意は同じ：距離・島の大きさ・形は“そのままの意味”ではなく、設定や乱数でも変わります。）

In [ ]:
# （さらに脱線）語数を増やして PCA / t-SNE / UMAP の 2D を比べる
import warnings
from sklearn.manifold import TSNE

BIG = {
    "countries":  ["japan","china","korea","india","france","germany","italy","brazil",
                   "russia","spain","canada","mexico","egypt","australia","thailand",
                   "vietnam","sweden","poland","greece","turkey"],
    "animals":    ["cat","dog","horse","elephant","tiger","rabbit","monkey","lion","wolf",
                   "bear","fox","deer","sheep","cow","pig","eagle","snake","frog","dolphin","owl"],
    "foods":      ["pizza","sushi","bread","rice","cheese","banana","coffee","noodle","curry",
                   "apple","potato","tomato","chocolate","soup","salad","butter","cake",
                   "sandwich","pasta","honey"],
    "sports":     ["soccer","football","basketball","baseball","tennis","golf","hockey","boxing",
                   "swimming","cycling","rugby","cricket","volleyball","skiing","surfing",
                   "wrestling","badminton","marathon"],
    "colors":     ["red","blue","green","yellow","purple","pink","black","white","brown",
                   "gray","gold","silver"],
    "jobs":       ["doctor","teacher","nurse","lawyer","engineer","scientist","farmer","chef",
                   "pilot","artist","writer","soldier","dentist","plumber","electrician",
                   "accountant","journalist","architect"],
    "vehicles":   ["car","truck","bus","train","bicycle","motorcycle","airplane","helicopter",
                   "boat","ship","taxi","subway","van","tractor","jet"],
    "instruments":["piano","guitar","violin","drums","flute","trumpet","saxophone","cello",
                   "harp","clarinet","trombone","accordion"],
    "clothing":   ["shirt","pants","dress","jacket","hat","shoes","socks","coat","sweater",
                   "skirt","gloves","scarf","jeans","boots"],
    "body":       ["head","hand","foot","arm","leg","eye","ear","nose","mouth","finger",
                   "knee","shoulder","elbow","neck"],
}
catcolors = {cat: plt.cm.tab10.colors[i] for i, cat in enumerate(BIG)}

words, labels = [], []
for cat, ws in BIG.items():
    for w in ws:
        if w in kv:
            words.append(w); labels.append(cat)
print(f"使用語数 = {len(words)} 語 / {len(BIG)} カテゴリ")
X = np.array([kv[w] for w in words])

with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    import umap
    layouts = {
        "PCA":   PCA(n_components=2, random_state=0).fit_transform(X),
        "t-SNE": TSNE(n_components=2, perplexity=30, init="pca",
                      random_state=0).fit_transform(X),
        "UMAP":  umap.UMAP(n_components=2, n_neighbors=15, min_dist=0.2,
                           random_state=0).fit_transform(X),
    }

def plot_big(name, P, save):
    fig, ax = plt.subplots(figsize=(7.6, 6.4))
    for cat in BIG:
        idx = [i for i, l in enumerate(labels) if l == cat]
        ax.scatter(P[idx, 0], P[idx, 1], s=28, color=catcolors[cat], label=cat)
    ax.set_title(f"Many words (~{len(words)}) — {name} (2D)", fontsize=13)
    ax.set_xticks([]); ax.set_yticks([])
    ax.legend(loc="best", fontsize=7, ncol=2, framealpha=0.9)
    fig.tight_layout()
    finish(fig, save)

plot_big("PCA",   layouts["PCA"],   "09_many_pca")
plot_big("t-SNE", layouts["t-SNE"], "10_many_tsne")
plot_big("UMAP",  layouts["UMAP"],  "11_many_umap")

## 補足②：単語ベクトルの統計

単語ベクトルの向きは単語の意味を表します。それでは単語ベクトルの長さは何を表すでしょうか？　また、単語間のコサインは意味の関連性の強さを表しますが、どのくらいの値なら関連が強いといえるのでしょうか？

### ノルムのヒストグラム、ランダムペアのコサインのヒストグラム
2つの分布を見ます：(1) 各ベクトルの長さ **‖x‖**、(2) **ランダムに選んだ2語のコサイン**（全語彙だと重いのでランダムに数万サンプル）。

**理論メモ**：もしベクトルが原点まわりに等方的に散らばっていれば、ランダムな2語のコサインは
**平均 0・分散 1/d**（d = 次元 = 300、std ≈ 1/√300 ≈ 0.058）の正規分布になります。
実際の単語ベクトルは **平均ベクトルが 0 からズレている（共通成分がある）** ため、
**中心化しないとコサインは正の側にずれます**。平均を引く（中心化）と理論分布に近づきます。

In [ ]:
# (1) ノルム ‖x‖ の分布、(2) ランダム2語の cos の分布（raw / 中心化 / 理論 N(0,1/d)）
rng = np.random.default_rng(0)
V = kv.vectors                      # (語彙数, 次元)
nv, d = V.shape

# (1) ‖x‖ ヒストグラム（ランダム 3万語）
samp = rng.choice(nv, size=30000, replace=False)
norms = np.linalg.norm(V[samp], axis=1)

# (2) ランダム 20万ペアの cos（raw=中心化なし / centered=平均を引く）
N = 200000
ia = rng.integers(0, nv, N); ib = rng.integers(0, nv, N)
ok = ia != ib; ia, ib = ia[ok], ib[ok]
A, B = V[ia], V[ib]
cos_raw = np.sum(A * B, axis=1) / (np.linalg.norm(A, axis=1) * np.linalg.norm(B, axis=1))
mu = V.mean(axis=0)                 # 全語彙の平均ベクトル
Ac, Bc = A - mu, B - mu
cos_cen = np.sum(Ac * Bc, axis=1) / (np.linalg.norm(Ac, axis=1) * np.linalg.norm(Bc, axis=1))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].hist(norms, bins=60, color="#4C72B0", alpha=0.85)
axes[0].axvline(norms.mean(), color="#C44E52", lw=2, label=f"mean = {norms.mean():.2f}")
axes[0].axvline(np.median(norms), color="#55A868", lw=2, ls="--", label=f"median = {np.median(norms):.2f}")
axes[0].plot([], [], " ", label=f"std = {norms.std():.2f}")
axes[0].set_title(f"distribution of vector norm  ||x||   (random {len(samp):,} words)", fontsize=12)
axes[0].set_xlabel("||x||"); axes[0].set_ylabel("count"); axes[0].legend()

ax = axes[1]
ax.hist(cos_raw, bins=80, density=True, color="#DD8452", alpha=0.7,
        label=f"raw (no centering)  mean={cos_raw.mean():+.3f}, std={cos_raw.std():.3f}")
ax.hist(cos_cen, bins=80, density=True, color="#55A868", alpha=0.6,
        label=f"centered  mean={cos_cen.mean():+.3f}, std={cos_cen.std():.3f}")
xs = np.linspace(-0.35, 0.6, 400)
ax.plot(xs, np.exp(-xs**2 / (2 * (1.0/d))) / np.sqrt(2 * np.pi * (1.0/d)),
        "k--", lw=1.6, label=f"theory N(0, 1/d),  d={d}  (std={1/np.sqrt(d):.3f})")
ax.axvline(0, color="0.5", lw=0.8)
ax.set_xlim(-0.35, 0.6)
ax.set_title(f"cosine of random word pairs   (random {len(A):,} pairs)", fontsize=12)
ax.set_xlabel("cosine similarity"); ax.set_ylabel("density"); ax.legend(fontsize=9)
fig.tight_layout()
finish(fig, "12_stats_norm_cos")

# raw cos の上側パーセンタイル：「どのくらいの cos なら“まれ”か」の目安
tops = [(50, "50%"), (10, "10%"), (1, "1%"), (0.1, "0.1%"), (0.01, "0.01%"), (0.001, "0.001%"), (0.0001, "0.0001%")]
vals = np.percentile(cos_raw, [100 - t for t, _ in tops])
print(f"ランダムな2語ペアの cos（中心化なし, {len(cos_raw):,} ペア）の上側パーセンタイル")
for (t, lab), v in zip(tops, vals):
    note = "   ← 参考（サンプル少・ばらつき大）" if t <= 0.0001 else ""
    print(f"  上位 {lab:>7}  →  cos ≥ {v:.3f}{note}")
print(f"  参考: cos(cat,dog)={kv.similarity('cat','dog'):.2f}, "
      f"cos(movie,film)={kv.similarity('movie','film'):.2f} "
      f"→ 上位0.01%（{vals[4]:.2f}）を大きく超える＝ランダムではまず起きない"
      f"（＝我々の例は“いい例”を選んだ証拠）")

### 代表ペアは分布のどこに来る？

これまで見た代表的なペアの cos を、ランダムペアの分布（中心化なし）に重ねます。
**関連の強いペアは右の裾**＝ランダムではめったに出ない位置にあります。
裏を返すと、cat–dog や movie–film のような分かりやすい例は、
**私たちが“いい例”を選んだ（チェリーピッキング）証拠**でもあります（笑）。

In [ ]:
# 代表ペアの cos を、ランダムペア分布（raw=cos_raw）の上に tick で重ねる
PAIRS_TICK = [
    # 無関係・普通のペア（cos が小さい＝山の中）
    ("cat", "economics"), ("tokyo", "banana"), ("flower", "computer"), ("king", "banana"),
    ("river", "coffee"), ("cat", "car"), ("dog", "car"),
    # 関連の強いペア（cos が大きい＝右の裾）
    ("happy", "sad"), ("tokyo", "osaka"), ("king", "queen"), ("cat", "dog"),
    ("man", "woman"), ("car", "truck"), ("increase", "decrease"), ("movie", "film"),
]
ticks = sorted((float(kv.similarity(a, b)), f"{a}-{b}") for a, b in PAIRS_TICK)

fig, ax = plt.subplots(figsize=(14, 6.5))
n, _, _ = ax.hist(cos_raw, bins=120, color="#9db2dd", alpha=0.85)
ymax = n.max()
ax.set_xlim(-0.25, 0.95)
ax.set_ylim(0, ymax * 1.04)                       # 余白を作らず棒で埋める
levels = [0.93, 0.76, 0.59, 0.42, 0.25]           # ラベルは下の方まで分散（棒と重なってOK）
for k, (c, lab) in enumerate(ticks):
    col = "#C44E52" if c >= 0.4 else "#3f3f3f"    # 関連強=赤 / 無関係=灰
    ax.axvline(c, color=col, lw=1.6, alpha=0.8)
    ax.annotate(f"{lab} {c:.2f}", xy=(c, levels[k % len(levels)] * ymax),
                ha="center", va="center", fontsize=9.5, color=col, fontweight="bold",
                bbox=dict(boxstyle="round,pad=0.22", fc="white", ec=col, alpha=0.92))
ax.set_xlabel("cosine similarity"); ax.set_ylabel("count (random pairs)")
ax.set_title(f"Where do our example pairs fall? — cosine of {len(cos_raw):,} "
             "random word pairs (raw)", fontsize=13)
fig.tight_layout()
finish(fig, "13_cos_with_examples")

### index（添字）と長さの関係

横軸は単語の **index（添字）** ＝`kv.index_to_key` の番号です。
GloVe 本家の `vocab_count` は語彙を **出現回数の降順にソート**して出力し
（ソース `src/vocab_count.c` の比較関数 `CompareVocab` が count の降順・`qsort`）、
学習済みベクトルもその語彙順で書き出されます。
このコードからの推測として、この index は **おそらく頻度ランク**（学習コーパス Wikipedia + Gigaword の頻度順）だと考えられます。
（※ データ自体に「ランク」と明記があるわけではなく、コードからの推測です。）
出典: GloVe リポジトリ <https://github.com/stanfordnlp/GloVe/blob/master/src/vocab_count.c>。

**図の見方**：左＝index 小（おそらく高頻度）。**index 小ほど ‖x‖ が小さく**、index 大になるほど長く、最末尾で再び短くなります。

In [ ]:
# (オプション) 単語の index（添字）と ‖x‖ の関係
# GloVe は語彙を出現回数の降順で出力する（src/vocab_count.c）ので、index は「おそらく頻度ランク」
print("先頭15語（GloVe は出現回数の降順で出力 → おそらく頻度順）:", kv.index_to_key[:15])
idx = rng.choice(nv, size=8000, replace=False)
nrm = np.linalg.norm(V[idx], axis=1)
fig, ax = plt.subplots(figsize=(8.5, 5))
ax.scatter(idx + 1, nrm, s=5, alpha=0.18, color="#4C72B0")
ax.set_xscale("log")
bins = np.logspace(0, np.log10(nv), 25)
which = np.digitize(idx + 1, bins)
bx, by = [], []
for k in range(1, len(bins)):
    sel = which == k
    if sel.sum() > 5:
        bx.append(np.median((idx + 1)[sel])); by.append(np.median(nrm[sel]))
ax.plot(bx, by, color="#C44E52", lw=2.5, label="median")
ax.set_xlabel("word index  (probably ~ frequency rank; small index = more frequent)")
ax.set_ylabel("||x||")
ax.set_title("word index (probably ~ frequency rank) vs vector norm ||x||", fontsize=12)
ax.legend(); ax.grid(alpha=0.25)
fig.tight_layout()
finish(fig, "14_freq_vs_norm")

## 3. 意味の足し算・引き算（アナロジー）

**結論**：`a : b :: c : ?` の答えは **b − a + c** で求まる。
`man : king :: woman : ?` → **queen**（男にとっての king＝王 は、女にとっての queen＝女王）。**成功例だけでなく、うまくいかない例も**一緒に見る
（埋め込みの限界の伏線）。

そのあとの図は king/man/woman/queen を「**性差の軸（横）**」と「**王位の軸（縦）**」に射影したもの。
2組の矢印が平行な平行四辺形になる＝引き算がきれいな方向をもつ。

In [ ]:
# (ラベル, a, b, c, 期待される答え)  意味: a:b :: c:expected  （答え = b - a + c）
ANALOGIES = [
    ("gender (royalty)", "man", "king", "woman", "queen"),
    ("gender (family)",  "man", "father", "woman", "mother"),
    ("capital FR->IT",   "france", "paris", "italy", "rome"),
    ("capital (Tokyo)",  "japan", "tokyo", "france", "paris"),
    ("capital JP->DE",   "japan", "tokyo", "germany", "berlin"),
    ("capital CN->JP",   "china", "beijing", "japan", "tokyo"),
    ("capital KR",       "japan", "tokyo", "korea", "seoul"),
    ("plural",           "car", "cars", "apple", "apples"),
    ("past tense",       "walk", "walked", "go", "went"),
    ("comparative",      "good", "better", "bad", "worse"),
    ("MISS (loose)",     "fish", "water", "bird", "air"),   # ゆるい関係は不安定
]

def rank_of(a, b, c, expected, search=50):
    res = kv.most_similar(positive=[b, c], negative=[a], topn=search)
    for i, (w, _) in enumerate(res, 1):
        if w == expected:
            return i
    return None

print("a : b  ::  c : ?     （答え = b - a + c）\n")
for label, a, b, c, expected in ANALOGIES:
    top = kv.most_similar(positive=[b, c], negative=[a], topn=5)
    top1 = top[0][0]
    r = rank_of(a, b, c, expected)
    mark = "OK" if top1 == expected else ("~ " if r and r <= 5 else "X ")
    top5 = ", ".join(f"{w}({s:.2f})" for w, s in top)
    print(f"[{mark}] {a} : {b} :: {c} : ?   ->  {top5}")
    print(f"       期待 '{expected}' は rank={r}\n")

In [ ]:
# king/man/woman/queen を「性差の軸」「王位の軸」に射影して平行四辺形を描く。
# （4語だけの PCA だと軸がアナロジー方向に揃わず台形に見えるため、多ペア平均で軸を作る。）
def _unit(x):
    return x / (np.linalg.norm(x) or 1.0)

gender = _unit(np.mean(
    [kv[f] - kv[m] for m, f in
     [("man", "woman"), ("king", "queen"), ("boy", "girl"), ("uncle", "aunt")]],
    axis=0))
status = np.mean([kv["king"] - kv["man"], kv["queen"] - kv["woman"]], axis=0)
status = _unit(status - (status @ gender) * gender)   # 性差成分を除いて直交化

words = ["man", "woman", "king", "queen"]
pos = {w: np.array([kv[w] @ gender, kv[w] @ status]) for w in words}

fig, ax = plt.subplots(figsize=(6.6, 5.6))
for w, (x, y) in pos.items():
    ax.scatter(x, y, s=90, color="#C44E52", zorder=3)
    ax.annotate(w, (x, y), fontsize=14, fontweight="bold",
                xytext=(7, 6), textcoords="offset points")

def arrow(w1, w2, color):
    ax.annotate("", xy=pos[w2], xytext=pos[w1],
                arrowprops=dict(arrowstyle="->", color=color, lw=2.2))

arrow("man", "woman", "#4C72B0")   # 性差（横・平行）
arrow("king", "queen", "#4C72B0")
arrow("man", "king", "#55A868")    # 王位（縦・平行）
arrow("woman", "queen", "#55A868")
ax.set_title("king − man + woman ≈ queen\n"
             "blue = gender, green = royalty — both pairs of arrows are parallel",
             fontsize=12)
ax.set_xlabel("← male      gender direction      female →")
ax.set_ylabel("← commoner    royalty direction    royal →")
ax.grid(alpha=0.25)
ax.margins(0.18)
fig.tight_layout()
finish(fig, "15_analogy_parallelogram")

## 4. 「関係」は一定方向のベクトル：国 → 首都

**結論**：どの国→首都ペアも **ほぼ同じ向きのベクトル**。2次元に落とすと矢印が平行にそろい、
欧州・アジアの地理的なまとまりも見える。

In [ ]:
PAIRS = [  # 欧州・アジアを混在
    ("france", "paris"), ("italy", "rome"), ("germany", "berlin"),
    ("russia", "moscow"), ("spain", "madrid"), ("egypt", "cairo"),
    ("japan", "tokyo"), ("china", "beijing"), ("korea", "seoul"),
    ("india", "delhi"), ("thailand", "bangkok"),
]
words = [w for pair in PAIRS for w in pair]
P = PCA(n_components=2, random_state=0).fit_transform(np.array([kv[w] for w in words]))
pos = {w: P[i] for i, w in enumerate(words)}

fig, ax = plt.subplots(figsize=(7.6, 6.0))
for country, capital in PAIRS:
    ax.scatter(*pos[country], s=70, color="#4C72B0", zorder=3)
    ax.scatter(*pos[capital], s=70, color="#C44E52", zorder=3)
    ax.annotate(country, pos[country], fontsize=11,
                xytext=(5, 5), textcoords="offset points", color="#27406b")
    ax.annotate(capital, pos[capital], fontsize=11,
                xytext=(5, 5), textcoords="offset points", color="#7a2b30")
    ax.annotate("", xy=pos[capital], xytext=pos[country],
                arrowprops=dict(arrowstyle="->", color="0.5", lw=1.5))
ax.scatter([], [], color="#4C72B0", label="country")
ax.scatter([], [], color="#C44E52", label="capital")
ax.legend(loc="best")
ax.set_title("country → capital is (roughly) the same vector for every pair",
             fontsize=12)
ax.set_xlabel("PCA dim 1"); ax.set_ylabel("PCA dim 2")
ax.grid(alpha=0.25)
fig.tight_layout()
finish(fig, "16_capitals")

## 5. 同じく一定方向：男 → 女

**結論**：man→woman, king→queen, … の「男→女」も **一定方向**（矢印が平行）。
04・05 と同じ「関係＝方向」の話。
（見どころ：king↔queen は man↔woman から少し離れる＝あくまで近似。prince–princess は king–queen のすぐ近くに来る。
また **male–female は cos≈0.89 とほぼ同義**で図でも最短の矢印になり、man→woman の「性差の方向」とは別物、というのも面白い。）

In [ ]:
GENDER_PAIRS = [("man", "woman"), ("king", "queen"), ("prince", "princess"),
                ("uncle", "aunt"), ("brother", "sister"), ("boy", "girl"),
                ("actor", "actress"), ("nephew", "niece"), ("he", "she"),
                ("male", "female")]
words = [w for pair in GENDER_PAIRS for w in pair]
P = PCA(n_components=2, random_state=0).fit_transform(np.array([kv[w] for w in words]))
pos = {w: P[i] for i, w in enumerate(words)}

fig, ax = plt.subplots(figsize=(7.6, 6.2))
for male, female in GENDER_PAIRS:
    ax.scatter(*pos[male], s=70, color="#4C72B0", zorder=3)
    ax.scatter(*pos[female], s=70, color="#C44E52", zorder=3)
    ax.annotate(male, pos[male], fontsize=10, xytext=(4, 4),
                textcoords="offset points", color="#27406b")
    ax.annotate(female, pos[female], fontsize=10, xytext=(4, 4),
                textcoords="offset points", color="#7a2b30")
    ax.annotate("", xy=pos[female], xytext=pos[male],
                arrowprops=dict(arrowstyle="->", color="0.5", lw=1.5))
ax.scatter([], [], color="#4C72B0", label="male")
ax.scatter([], [], color="#C44E52", label="female")
ax.legend(loc="best")
ax.set_title("male → female is a consistent direction (parallel arrows)", fontsize=12)
ax.set_xlabel("PCA dim 1"); ax.set_ylabel("PCA dim 2")
ax.grid(alpha=0.25)
fig.tight_layout()
finish(fig, "17_gender")

### （おまけ）3次元で見るとどうなる？

2次元の図は、本当は300次元のベクトルを2次元に押しつぶした「影」です。
同じ男女ペアを **3次元（PCA の PC1–PC2–PC3）** で表示してみます。
（静止画なので回せませんが、奥行き＝PC3 が加わると、2次元では重なって見えた点の前後関係が分かります。）

In [ ]:
# （おまけ）同じ男女ペアを PCA で3次元（PC1–PC2–PC3）に落として見る
PAIRS_3D = [("man", "woman"), ("king", "queen"), ("prince", "princess"),
            ("uncle", "aunt"), ("brother", "sister"), ("boy", "girl"),
            ("actor", "actress"), ("nephew", "niece"), ("he", "she"),
            ("male", "female")]
words = [w for pair in PAIRS_3D for w in pair]
P3 = PCA(n_components=3, random_state=0).fit_transform(np.array([kv[w] for w in words]))
pos = {w: P3[i] for i, w in enumerate(words)}

fig = plt.figure(figsize=(8, 6.6))
ax = fig.add_subplot(projection="3d")
for male, female in PAIRS_3D:
    xm, ym, zm = pos[male]; xf, yf, zf = pos[female]
    ax.scatter(xm, ym, zm, s=55, color="#4C72B0", depthshade=False, zorder=3)
    ax.scatter(xf, yf, zf, s=55, color="#C44E52", depthshade=False, zorder=3)
    ax.plot([xm, xf], [ym, yf], [zm, zf], color="0.6", lw=1.2)
    ax.text(xm, ym, zm, " " + male, fontsize=8, color="#27406b")
    ax.text(xf, yf, zf, " " + female, fontsize=8, color="#7a2b30")
ax.legend(handles=[mpatches.Patch(color="#4C72B0", label="male"),
                   mpatches.Patch(color="#C44E52", label="female")], loc="upper left")
ax.set_xlabel("PC1"); ax.set_ylabel("PC2"); ax.set_zlabel("PC3")
ax.set_title("male → female pairs — PCA (3D)", fontsize=13)
ax.view_init(elev=18, azim=-60)
ax.set_box_aspect(None, zoom=0.85)
finish(fig, "18_gender_3d")

---
## 埋め込みの「限界」編

ここからは、単語ベクトルが **うまくいかない / 誤解を生む** 例です。
便利な道具ですが、**何が苦手か**を知っておくことが大切です。

## 限界①：多義語は意味が混ざる（1単語 = 1ベクトル）

**結論**：1つの単語に1つのベクトルしか割り当てないので、**複数の意味をもつ語は意味が混ざる**。
`mouse`（動物＋PCの装置）、`amazon`（企業＋熱帯雨林）、`python`（コメディ＋言語）、
`windows`（窓＋OS）の近傍を意味ごとに色分けすると、両方の意味の語が混在する。

In [ ]:
# word -> ((意味A ラベル, 語の集合A), (意味B ラベル, 語の集合B))
POLY = {
    "mouse":   (("animal", {"mice", "rat", "rabbit", "rodent", "monkey", "rats"}),
                ("computer device", {"keyboard", "joystick"})),
    "amazon":  (("company", {"amazon.com", "kindle", "itunes", "bezos"}),
                ("nature/river", {"rainforest", "amazonian", "jungle", "deforestation"})),
    "python":  (("comedy (Monty Python)", {"monty", "cleese", "pythons", "grail", "skit"}),
                ("programming language", {"perl", "php", "scripting"})),
    "windows": (("opening in a wall", {"window", "doors"}),
                ("operating system", {"xp", "microsoft", "desktop", "browser",
                                       "software", "macintosh"})),
}
CA, CB, COTHER = "#4C72B0", "#DD8452", "0.6"

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, (word, ((la, sa), (lb, sb))) in zip(axes.flat, POLY.items()):
    pairs = kv.most_similar(word, topn=8)[::-1]
    words = [w for w, _ in pairs]
    sims = [s for _, s in pairs]
    colors = [CA if w in sa else CB if w in sb else COTHER for w in words]
    ax.barh(words, sims, color=colors)
    ax.set_xlim(0, 0.85)
    ax.set_xlabel("cosine similarity")
    ax.set_title(f"neighbours of “{word}”", fontsize=12)
    for y, s in enumerate(sims):
        ax.text(s + 0.008, y, f"{s:.2f}", va="center", fontsize=8, color="0.3")
    ax.legend(handles=[mpatches.Patch(color=CA, label=la),
                       mpatches.Patch(color=CB, label=lb)],
              loc="lower right", fontsize=9, framealpha=0.95).set_zorder(5)
fig.suptitle("One word = one vector → the two meanings get mixed together (polysemy)",
             fontsize=14)
fig.tight_layout(rect=(0, 0, 1, 0.96))
finish(fig, "19_polysemy")

## 限界②：反意語は「近い」

**結論**：反対の意味の語は **同じ文脈に出る**（「天気が暑い/寒い」）ので、ベクトルは近い。
無関係な語と比べると、反意語のコサインがずっと高い。
（図の反意語ペア：increase–decrease＝増加・減少, up–down＝上下, fast–slow＝速い・遅い, good–bad＝良い・悪い, happy–sad＝うれしい・悲しい, big–small＝大きい・小さい。右の灰色は比較用の無関係語。）
本質は「同じ"良し悪し軸"の語＝向きは逆でも軸は同じ」。

In [ ]:
# anchor -> (反意語, 無関係な語＝ベースライン)
ITEMS = [
    ("increase", "decrease", "purple"),
    ("up", "down", "banana"),
    ("fast", "slow", "chair"),
    ("good", "bad", "car"),
    ("happy", "sad", "concrete"),
    ("big", "small", "yellow"),
]
C_ANT, C_BASE = "#C44E52", "0.6"

anchors = [a for a, _, _ in ITEMS]
ant = [kv.similarity(a, b) for a, b, _ in ITEMS]
base = [kv.similarity(a, c) for a, _, c in ITEMS]

y = np.arange(len(anchors)); h = 0.38
fig, ax = plt.subplots(figsize=(9, 5.6))
ax.barh(y + h / 2, ant, height=h, color=C_ANT, label="antonym (opposite meaning)")
ax.barh(y - h / 2, base, height=h, color=C_BASE, label="unrelated word")
for i, (a, b, c) in enumerate(ITEMS):
    ax.text(ant[i] + 0.01, y[i] + h / 2, f"{b}  {ant[i]:.2f}",
            va="center", fontsize=9, color="#7a2b30")
    ax.text(base[i] + 0.01, y[i] - h / 2, f"{c}  {base[i]:.2f}",
            va="center", fontsize=9, color="0.4")
ax.set_yticks(y); ax.set_yticklabels(anchors, fontsize=11)
ax.set_xlim(min(0, min(base)) - 0.02, 0.95)
ax.axvline(0, color="0.7", lw=0.8)
ax.set_xlabel("cosine similarity to the anchor word")
ax.set_title("Antonyms are CLOSE, not far\n"
             "(same 'good–bad' axis; the vector keeps the topic, not the polarity)",
             fontsize=12)
ax.legend(loc="lower right")
fig.tight_layout()
finish(fig, "20_antonyms")

## 限界③：ことばのベクトルに埋め込まれた社会的バイアス

単語ベクトルは大量の文章（Wikipedia 等）から作られるため、**その文章に含まれる社会の偏り
（ジェンダーなどのバイアス）をそのまま吸収**します。以下は埋め込みが **偏見を再生産してしまう**
例で、**この偏りを肯定するためではなく、注意すべき問題として** 示します。

**結論（その1）**：性別と無関係なはずの **職業語が「性差の方向」に沿って並びます**。
06 で見た「男→女」方向に各職業語を射影（その向きにどれだけ傾くかを測ること）すると、nurse（看護師）/ maid（お手伝い）… が女性側、
engineer（技術者）/ boss（上司）… が男性側に偏ります。
（図のその他の職業語：receptionist＝受付, housekeeper＝家政婦, nanny＝子守, librarian＝司書,
surgeon＝外科医, mechanic＝整備士, carpenter＝大工, architect＝建築家。）

また、次の「バイアス その2」で出てくる **doctor と physician（医師）はこの軸でほぼ中央**、**engineer より technician（技師）の方が中央寄り**、というのもこの図で確認できます。

In [ ]:
# 06 と同じ「性差の方向」（女 - 男 を複数ペアで平均）。正の向き = 女性側。
GENDER_PAIRS = [("man", "woman"), ("king", "queen"), ("boy", "girl"),
                ("uncle", "aunt"), ("brother", "sister"), ("he", "she")]
g = np.mean([kv[f] - kv[m] for m, f in GENDER_PAIRS], axis=0)
g = g / (np.linalg.norm(g) or 1.0)

# 職業語（すべて GloVe の小文字語彙）。"secretary" は corpus 上 "Secretary of State"
# 等の政治職に強く引かれて男性側に出るため、話が濁るので外している。
OCCUPATIONS = [
    "nurse", "receptionist", "nanny", "housekeeper", "maid",
    "teacher", "librarian", "dancer",
    "doctor", "physician", "engineer", "technician", "programmer",
    "mechanic", "boss", "scientist", "surgeon", "soldier", "architect", "carpenter",
]
C_FEMALE, C_MALE = "#C44E52", "#4C72B0"

scored = []
for w in OCCUPATIONS:
    v = kv[w]
    scored.append((w, float((v / (np.linalg.norm(v) or 1.0)) @ g)))
scored.sort(key=lambda t: t[1])
words = [w for w, _ in scored]; vals = [s for _, s in scored]
colors = [C_FEMALE if s > 0 else C_MALE for s in vals]

lim = max(abs(min(vals)), abs(max(vals))) * 1.35
fig, ax = plt.subplots(figsize=(8.5, 7.8))
ax.barh(words, vals, color=colors)
ax.axvline(0, color="0.4", lw=1.0)
ax.set_xlim(-lim, lim)
for y, s in enumerate(vals):
    ax.text(s + (0.004 if s >= 0 else -0.004), y, f"{s:+.2f}",
            va="center", ha="left" if s >= 0 else "right", fontsize=8, color="0.3")
ax.set_xlabel("←  male direction        gender axis        female direction  →")
ax.set_title("Occupation words carry gender bias\n"
             "projection onto the she − he direction — the embedding inherits "
             "stereotypes from its training text", fontsize=12)
ax.grid(axis="x", alpha=0.25)
fig.tight_layout()
finish(fig, "21_bias_projection")

### バイアス その2：アナロジーが偏見を再生産する

**結論（その2）**：`king − man + woman ≈ queen` と同じ計算を職業に使うと、**偏ったことば** が上位に出ます。
ここでは「性差軸（女−男）への射影」（正＝女性側・負＝男性側、図09と同じ尺度）で偏りの向きも確かめます。

**例1: `man : doctor :: woman : ?`**（「男にとっての doctor（医者）は、女にとっては？」）
→ 上位は physician（医師）, **nurse（看護師）**, **pregnant（妊娠した）**, dentist（歯科医）…
- physician（医師）は doctor とほぼ同義。性差軸では doctor ≈ +0.02 / physician ≈ −0.05 で、
  physician はむしろ**ごくわずか男性寄り**。つまり physician が1位なのは「一番近い言い換え」が出ているだけで、ここ自体は強い偏りではありません。
- 偏りがはっきり出るのは、**女性側へ動かすと nurse（+0.29 と明確に女性寄り）や pregnant が上位に来る**こと——
  「女性に寄せると看護師・妊娠が出てくる」という、学習データの偏見の反映です。

**例2: `man : engineer :: woman : ?`**（engineer＝技術者）→ **technician（技師）**
- これは“微妙な偏り”が分かりやすい例。technician は engineer より少し地位が低く、男性度も薄い語で、
  実測でも engineer（−0.16）より technician（−0.03）の方が女性寄り。
  → `technician − engineer` は女性方向の成分をもち、**「女性版にすると“格下げ”される」**向きの偏りが乗っています。

**ポイント**：1位の語だけでなく **上位の並び全体** と、**語の置き換わり方（doctor→nurse、engineer→technician）** に偏りが表れます。
（数値は性差軸への射影 cos。`scratch/explore_limits.py` で再現できます。）

In [ ]:
# a:b :: c:?  （答え = b - a + c）。入力語の単純な複数形は答えではないので除外して上位を見る。
BIAS = [
    ("man", "doctor", "woman"),
    ("woman", "nurse", "man"),
    ("man", "boss", "woman"),
    ("man", "engineer", "woman"),
]
TOPN = 7

def _inflections(w):
    return {w, w + "s", w + "es", w.rstrip("s")}

def completions(a, b, c, keep=TOPN):
    banned = _inflections(a) | _inflections(b) | _inflections(c)
    out = []
    for w, s in kv.most_similar(positive=[b, c], negative=[a], topn=keep + 8):
        if w in banned:
            continue
        out.append((w, s))
        if len(out) >= keep:
            break
    return out

fig, axes = plt.subplots(2, 2, figsize=(12, 7.6))
for ax, (a, b, c) in zip(axes.flat, BIAS):
    pairs = completions(a, b, c)[::-1]
    words = [w for w, _ in pairs]; sims = [s for _, s in pairs]
    ax.barh(words, sims, color="#4C72B0")
    ax.set_xlim(0, 0.72)
    ax.set_xlabel("cosine similarity")
    ax.set_title(f"{a} : {b}   ::   {c} : ?", fontsize=12, family="monospace")
    for y, s in enumerate(sims):
        ax.text(s + 0.008, y, f"{s:.2f}", va="center", fontsize=8, color="0.3")
fig.suptitle("The same arithmetic behind “king − man + woman ≈ queen” "
             "fills occupations with stereotypes\n"
             "(woman → nurse / pregnant / girlfriend / mother): bias is reproduced "
             "from the training text, not invented", fontsize=12)
fig.tight_layout(rect=(0, 0, 1, 0.93))
finish(fig, "22_bias_analogy")

---
## 出典（参考文献）

このノートで使う GloVe ベクトルそのものの出典・ライセンスは、冒頭の
「**使用する単語ベクトルと出典**」を参照。以下は図で示した現象に関する主な文献です。

- **word2vec**：Mikolov, Chen, Corrado, Dean. *Efficient Estimation of Word Representations in Vector Space.* 2013.
  arXiv: <https://arxiv.org/abs/1301.3781>
  - アナロジー（`king − man + woman ≈ queen` 系）：Mikolov, Yih, Zweig.
    *Linguistic Regularities in Continuous Space Word Representations.* NAACL-HLT 2013.
    ACL Anthology: <https://aclanthology.org/N13-1090/>
- **GloVe**：Pennington, Socher, Manning. *GloVe: Global Vectors for Word Representation.* EMNLP 2014.
  ACL Anthology: <https://aclanthology.org/D14-1162/>
- **性別×職業バイアスの射影分析**：Bolukbasi, Chang, Zou, Saligrama, Kalai.
  *Man is to Computer Programmer as Woman is to Homemaker? Debiasing Word Embeddings.* NeurIPS (NIPS) 2016.
  （題意「男性がプログラマーなら、女性は主婦？」＝埋め込みに潜む性差バイアスと、その除去〔Debiasing＝バイアス除去〕を論じた研究）
  <https://proceedings.neurips.cc/paper/2016/hash/a486cd07e4ac3d270571622f4f316ec5-Abstract.html>

- 次元削減の手法（**PCA / t-SNE / UMAP**）の説明・文献・URL は、本文の「次元削減を“雰囲気”でつかむ」節に記載（重複を避けここでは省略）。